In [ ]:
import requests
import pandas as pd
import sqlite3

LOCATIONS = {
    "York": (53.96, -1.08),
    "Leeds": (53.80, -1.55),
    "Bristol": (51.45, -2.59),
    "Exeter": (50.72, -3.53),
}

# ---------- EXTRACT ----------
def extract(city, lat, lon):
    url = (
        "https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}&longitude={lon}"
        "&daily=temperature_2m_max,temperature_2m_min,precipitation_sum"
        "&past_days=14&forecast_days=1&timezone=Europe/London"
    )
    response = requests.get(url)
    response.raise_for_status()          # crash clearly if the API fails
    df = pd.DataFrame(response.json()["daily"])
    df["city"] = city
    return df


In [ ]:
test = extract("York", 53.96, -1.08)
print(test.columns)
test.head()

Index(['time', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum',
       'city'],
      dtype='object')


,time,temperature_2m_max,temperature_2m_min,precipitation_sum,city
0,2026-10-31,12.0,6.4,0.0,York
1,2026-11-01,11.6,4.2,0.0,York
2,2026-11-02,11.8,5.7,0.4,York
3,2026-11-03,11.5,8.5,0.1,York
4,2026-11-04,12.6,9.2,1.1,York


In [ ]:
# ---------- TRANSFORM ----------
def transform(df):
    df = df.copy()
    # TODO 1: rename columns to: date, max_temp, min_temp, rain_mm
    df = df.rename(columns={"time":"date","temperature_2m_max":"max_temp","temperature_2m_min":"min_temp","precipitation_sum":"rain_mm"})
    # TODO 2: convert 'date' to a proper date with pd.to_datetime
    df["date"] = pd.to_datetime(df["date"])
    # TODO 3: drop any rows with missing values
    df = df.dropna()
    # TODO 4: add a column 'rain_day': True if rain_mm > 1
    df["rainy_day"] = df["rain_mm"] > 1
    # TODO 5: add a column 'workable': True if rain_mm < 5 AND min_temp > 0
    df["workable"] = (df["rain_mm"] < 5) & (df["min_temp"] > 0)
    return df


In [ ]:
# ---------- LOAD ----------
def load(df, db_name="weather.db"):
    conn = sqlite3.connect(db_name)
    df.to_sql("daily_weather", conn, if_exists="replace", index=False)
    conn.close()



In [ ]:
# ---------- RUN THE PIPELINE ----------
frames = []
for city, (lat, lon) in LOCATIONS.items():
    frames.append(extract(city, lat, lon))

raw = pd.concat(frames, ignore_index=True)
clean = transform(raw)
load(clean)
print("Pipeline complete")

Pipeline complete


In [ ]:
conn = sqlite3.connect("weather.db")

In [ ]:
pd.read_sql("""
    SELECT city, SUM(workable) AS total_workable_days
    FROM daily_weather
    GROUP BY city
""", conn)

,city,total_workable_days
0,Bristol,14
1,Exeter,12
2,Leeds,15
3,York,15


In [ ]:
pd.read_sql("""
    SELECT city, SUM(rain_mm) AS total_rain
    FROM daily_weather
    GROUP BY city
    ORDER BY total_rain DESC
    LIMIT 1
""", conn)

,city,total_rain
0,Exeter,22.6


In [ ]:
pd.read_sql("""
    SELECT city, date AS coldest_day, min_temp
    FROM daily_weather
    ORDER BY min_temp ASC
    LIMIT 1
""", conn)

,city,coldest_day,min_temp
0,Exeter,2026-11-13 00:00:00,0.0


In [ ]:
pd.read_sql("""
    SELECT city, SUM(workable) AS workable_days, COUNT(*) AS total_days, SUM(workable) * 100 / COUNT(*) AS percentage
    FROM daily_weather
    GROUP BY city
    ORDER BY percentage
""", conn)

,city,workable_days,total_days,percentage
0,Exeter,12,15,80
1,Bristol,14,15,93
2,Leeds,15,15,100
3,York,15,15,100
